In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine, text, bindparam

# Set plotting defaults
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

# ==============================================================================
# 1. RETRIEVE ALL PA-ENDING EVENTS FROM DATABASE
# ==============================================================================
print("Querying full pitch logs for qualified hitters...")

# Extract qualified batter list from your existing fl_metrics dataframe
qual_batters = fl_metrics["batter"].unique().tolist()

SQL_PITCHES = text("""
    SELECT batter, year, pitch_call, k_or_bb, play_result, strikes, balls, rel_speed
    FROM pitches
    WHERE batter IN :batter_list
""").bindparams(bindparam("batter_list", expanding=True))

engine = get_engine() # Reuses engine configuration from previous step
with engine.connect() as conn:
    pitch_log = pd.read_sql(SQL_PITCHES, conn, params={"batter_list": qual_batters})

# Normalize types to ensure clean merging
pitch_log = pitch_log.dropna(subset=["year"]).copy()
pitch_log["year"] = pitch_log["year"].astype("int64")
pitch_log["batter"] = pitch_log["batter"].astype(str)

print(f"Pulled {len(pitch_log):,} pitch rows for {pitch_log['batter'].nunique()} batters.")

# ==============================================================================
# 2. ISOLATE PA-ENDING PLAYS & CALCULATE FULL wOBA VALUES
# ==============================================================================
# Identify specific play outcomes
pitch_log["is_k"] = pitch_log["k_or_bb"] == "strikeout"
pitch_log["is_bb"] = pitch_log["k_or_bb"] == "walk"
pitch_log["is_hbp"] = pitch_log["pitch_call"] == "hit_by_pitch"
pitch_log["is_pa_ending"] = (
    pitch_log["is_k"] | pitch_log["is_bb"] | pitch_log["is_hbp"] | (pitch_log["pitch_call"] == "in_play")
)

# Isolate to one record per plate appearance
pa_ending = pitch_log[pitch_log["is_pa_ending"]].copy()

# Normalize play result strings for mapping
pa_ending["pr_norm"] = pa_ending["play_result"].astype(str).str.lower().str.replace(" ", "")
pa_ending.loc[pa_ending["is_bb"], "pr_norm"] = "walk"
pa_ending.loc[pa_ending["is_hbp"], "pr_norm"] = "hit_by_pitch"

# Apply full sabermetric weights
WOBA_W_FULL = {
    "walk": 0.751, "hit_by_pitch": 0.786, "single": 0.935, 
    "double": 1.289, "triple": 1.525, "home_run": 2.146
}
pa_ending["woba_val_full"] = pa_ending["pr_norm"].map(WOBA_W_FULL).fillna(0.0)

# ==============================================================================
# 3. MERGE TRAITS AND ESTREABLISH BIN SEGMENTS
# ==============================================================================
# Format hitter metrics for merging
hitter_traits = fl_metrics[["batter", "year", "bat_speed_cal_mph", "PAA", "PAD_lac"]].copy()
hitter_traits["batter"] = hitter_traits["batter"].astype(str)
hitter_traits["year"] = hitter_traits["year"].astype("int64")

# Merge traits onto individual plate appearances
full_pa_data = pa_ending.merge(hitter_traits, on=["batter", "year"], how="inner")

# Group metrics into discrete quantile bins
full_pa_data["Bat_Speed_Bin"] = pd.qcut(
    full_pa_data["bat_speed_cal_mph"], q=5, 
    labels=["Very Low", "Low", "Average", "High", "Elite"]
)
full_pa_data["PAA_Bin"] = pd.qcut(
    full_pa_data["PAA"], q=3, 
    labels=["Low PAA (Flat)", "Medium PAA", "High PAA (Up)"]
)
full_pa_data["PAD_lac_Bin"] = pd.qcut(
    full_pa_data["PAD_lac"], q=3, 
    labels=["Opp/Center Focus", "Moderate Pull", "Heavy Pull"]
)

# ==============================================================================
# 4. ANALYSIS 1: wOBA VS wOBAcon BY BAT SPEED QUINTILES
# ==============================================================================
print("\n── Offense Profiles by Bat Speed Quintiles ──")

# Aggregate both overall wOBA and Contact-only wOBAcon
speed_profile = full_pa_data.groupby("Bat_Speed_Bin").apply(
    lambda g: pd.Series({
        "Total_PAs": len(g),
        "wOBA": g["woba_val_full"].mean(),
        "wOBAcon": g.loc[g["pitch_call"] == "in_play", "woba_val_full"].mean()
    })
).reset_index()

print(speed_profile.to_string(index=False))

# Reshape data to plot metrics side-by-side
speed_melted = pd.melt(
    speed_profile, id_vars=["Bat_Speed_Bin"], 
    value_vars=["wOBA", "wOBAcon"], 
    var_name="Metric", value_name="Value"
)

plt.figure(figsize=(10, 6))
sns.barplot(data=speed_melted, x="Bat_Speed_Bin", y="Value", hue="Metric", palette="Set2")
plt.title("Overall wOBA vs. wOBAcon by Hitter Bat Speed Tier", fontsize=14, fontweight="bold")
plt.xlabel("Bat Speed Quintile Tiers")
plt.ylabel("Value")
plt.ylim(0.200, 0.550)
plt.tight_layout()
plt.savefig("output/woba_vs_wobacon_batspeed.png")
plt.close()

# ==============================================================================
# 5. ANALYSIS 2: SIDE-BY-SIDE GEOMETRIC PRODUCTION MATRICES (PAA × PAD_lac)
# ==============================================================================
# Compare how swing mechanics map to damage-on-contact vs total run creation

geo_woba = full_pa_data.groupby(["PAA_Bin", "PAD_lac_Bin"]).agg(
    wOBA=("woba_val_full", "mean")
).reset_index().pivot(index="PAA_Bin", columns="PAD_lac_Bin", values="wOBA")

geo_wobacon = full_pa_data[full_pa_data["pitch_call"] == "in_play"].groupby(["PAA_Bin", "PAD_lac_Bin"]).agg(
    wOBAcon=("woba_val_full", "mean")
).reset_index().pivot(index="PAA_Bin", columns="PAD_lac_Bin", values="wOBAcon")

# Setup Heatmap Subplots
fig, axes = plt.subplots(1, 2, figsize=(20, 8), sharey=True)

# Heatmap A: Overall wOBA (Includes strikeouts/walks)
sns.heatmap(geo_woba, annot=True, fmt=".3f", cmap="magma", vmin=0.280, vmax=0.420, ax=axes[0], linewidths=.5)
axes[0].set_title("Overall Hitter wOBA Matrix (All Events)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Preferred Attack Direction (PAD_lac)")
axes[0].set_ylabel("Parabola Launch Angle Peak (PAA)")
axes[0].invert_yaxis()

# Heatmap B: wOBAcon (Contact only)
sns.heatmap(geo_wobacon, annot=True, fmt=".3f", cmap="magma", vmin=0.350, vmax=0.550, ax=axes[1], linewidths=.5)
axes[1].set_title("Hitter wOBAcon Matrix (BBE Contact Only)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Preferred Attack Direction (PAD_lac)")
axes[1].set_ylabel("")
axes[1].invert_yaxis()

plt.suptitle("Sabermetric Shape Optimization: All Plate Appearances vs. Contact Results", fontsize=16, fontweight="bold", y=0.98)
plt.tight_layout()
plt.savefig("output/woba_vs_wobacon_geometry_heatmaps.png")
plt.close()

print("\nProcessing complete! Evaluation charts exported to output directory:")
print("-> output/woba_vs_wobacon_batspeed.png")
print("-> output/woba_vs_wobacon_geometry_heatmaps.png")

Querying full pitch logs for qualified hitters...


NameError: name 'fl_metrics' is not defined